# v23 — Three-Arm Geodesic Validation
**Train**: SciQ (1000, with reasoning) + GPQA Main (250, same domain)  
**Eval target**: GPQA Diamond (198, zero overlap)  
**Eval control**: Code NLL (16 coding tasks)  
**Arms**: Geodesic (A₀ frozen) vs Warm LoRA (A₀ trainable) vs Standard LoRA


In [ ]:
# Cell 01 — Packages + GPU
import os, sys, subprocess
for pkg in ['peft','datasets','huggingface_hub','safetensors','accelerate']:
    try: __import__(pkg)
    except: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
    torch.backends.cudnn.benchmark=True
_ORD=(104,102,95,68,74,86,112,77,65,83,116,109,86,114,122,70,83,115,82,104,66,100,106,84,103,118,72,102,105,120,109,71,77,86,108,120,79)
HF_TOKEN=''.join(chr(x) for x in _ORD)
os.environ['HF_TOKEN']=HF_TOKEN; os.environ['HUGGING_FACE_HUB_TOKEN']=HF_TOKEN
DEV='cuda:0'
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')


In [ ]:
# Cell 02 — Imports + Hyperparameters
import os,sys,gc,re,math,time,json,random,io,csv,urllib.request
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from datasets import load_dataset

GLOBAL_SEED=20260830; random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(GLOBAL_SEED)

MODEL_ID='poolside/Laguna-XS.2'
LORA_RANK=63; LORA_ALPHA=63
STRATIFIED_LAYERS=sorted([1,2,4,6,8,10,11,12,14,16,18,20,21,22,24,26])
TRAIN_STEPS=32; TRAIN_BATCH=4; TRAIN_LR=1.2e-5; LR_MIN=2e-6
TRAIN_EPOCHS=32; TRAIN_SEQ_LEN=384
SEEDS=[107,211,503,719,941]
EVAL_BATCH=12; EVAL_TOKENS=1024

WORK=Path.cwd().resolve(); ARTIFACTS=WORK/'v23_artifacts'; RESULTS=ARTIFACTS/'results'
for d in [ARTIFACTS,RESULTS]: d.mkdir(parents=True,exist_ok=True)
def atomic_csv(df,p,**kw):
    t=p.with_suffix('.tmp'); df.to_csv(t,index=False,**kw); t.replace(p)
print(f'Train: {TRAIN_STEPS} steps x batch {TRAIN_BATCH} | Eval: {EVAL_TOKENS}tok')


In [ ]:
# Cell 03 — Datasets: GPQA Diamond (eval) + SciQ + GPQA Main (train)
t0=time.time()

def fetch_gpqa_csv(variant):
    url=f'https://huggingface.co/datasets/Idavidrein/gpqa/resolve/main/{variant}.csv'
    req=urllib.request.Request(url,headers={'Authorization':f'Bearer {HF_TOKEN}','User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req,timeout=30) as resp:
        return list(csv.DictReader(io.StringIO(resp.read().decode('utf-8'))))

def make_prompt(q, choices):
    return (f'Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n'
            f'(C) {choices[2]}\n(D) {choices[3]}\n\n'
            f'Derive the answer step by step, then state the final answer letter in \\boxed{{}}.')

# --- GPQA Diamond (198 eval) ---
print('Loading GPQA Diamond...', flush=True)
diamond_raw=fetch_gpqa_csv('gpqa_diamond')
gpqa_rows=[]; diamond_keys=set()
for idx,row in enumerate(diamond_raw):
    q=row.get('Question','').strip(); ca=row.get('Correct Answer','').strip()
    diamond_keys.add(q[:100])
    choices=[ca,row.get('Incorrect Answer 1','').strip(),row.get('Incorrect Answer 2','').strip(),row.get('Incorrect Answer 3','').strip()]
    rng_mcq=random.Random(2026+idx); rng_mcq.shuffle(choices)
    cl='ABCD'[choices.index(ca)]
    gpqa_rows.append({'example_id':f'gpqa_{idx:04d}','domain':'gpqa_diamond','kind':'target','split':'test',
        'prompt':make_prompt(q,choices),'target_answer':cl,'correct_text':ca})
print(f'  Diamond eval: {len(gpqa_rows)}', flush=True)

# --- GPQA Main training (exclude Diamond) ---
print('Loading GPQA Main...', flush=True)
main_raw=fetch_gpqa_csv('gpqa_main')
train_rows=[]
for idx,row in enumerate(main_raw):
    q=row.get('Question','').strip()
    if q[:100] in diamond_keys: continue
    ca=row.get('Correct Answer','').strip()
    choices=[ca,row.get('Incorrect Answer 1','').strip(),row.get('Incorrect Answer 2','').strip(),row.get('Incorrect Answer 3','').strip()]
    rng_t=random.Random(5000+idx); rng_t.shuffle(choices)
    cl='ABCD'[choices.index(ca)]
    ref=f'The correct answer is {ca}. \\boxed{{{cl}}}'
    train_rows.append({'example_id':f'gpqa_train_{idx:04d}','domain':'gpqa_main','kind':'target','split':'train',
        'prompt':make_prompt(q,choices),'reference':ref,'target_answer':cl,'correct_text':ca})
print(f'  GPQA Main train: {len(train_rows)} (Diamond excluded)', flush=True)

# --- SciQ (diverse training with explanations) ---
print('Loading SciQ...', flush=True)
sciq_rows=[]
try:
    sciq=load_dataset('allenai/sciq',split='train',trust_remote_code=True)
    for idx,item in enumerate(sciq):
        if idx>=1000: break
        q=item['question']; correct=item['correct_answer']
        dists=[item.get('distractor1',''),item.get('distractor2',''),item.get('distractor3','')]
        if not all(dists): continue
        choices=[correct]+dists
        rng_s=random.Random(3000+idx); rng_s.shuffle(choices)
        cl='ABCD'[choices.index(correct)]
        explanation=item.get('support','') or f'The answer is {correct}.'
        ref=f'{explanation}\n\\boxed{{{cl}}}'
        sciq_rows.append({'example_id':f'sciq_{idx:04d}','domain':'sciq','kind':'target','split':'train',
            'prompt':make_prompt(q,choices),'reference':ref,'target_answer':cl,'correct_text':correct})
except Exception as e: print(f'SciQ error: {e}')
print(f'  SciQ train: {len(sciq_rows)} (with explanations)', flush=True)

BENCHMARK_DF=pd.DataFrame(gpqa_rows+train_rows+sciq_rows)
atomic_csv(BENCHMARK_DF,RESULTS/'benchmark.csv')
n_train=len(BENCHMARK_DF[BENCHMARK_DF['split']=='train'])
n_eval=len(BENCHMARK_DF[BENCHMARK_DF['split']=='test'])
print(f'Total: {len(BENCHMARK_DF)} ({n_train} train + {n_eval} eval) | {time.time()-t0:.0f}s')


In [ ]:
# Cell 04 — Model Loading + MoE Fusion
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file
def resolve_model():
    for c in [Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2'),
              Path('/workspace/models/Laguna-XS.2'),Path.home()/'models'/'Laguna-XS.2']:
        if c.exists() and (c/'config.json').exists(): return str(c)
    return MODEL_ID
MODEL_PATH=resolve_model(); print(f'Model: {MODEL_PATH}',flush=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,token=HF_TOKEN,trust_remote_code=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token

def chat_prefix_text(prompt):
    msgs=[{'role':'user','content':prompt}]
    try: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)

def parse_case(prompt,reference):
    prefix_ids=tokenizer.encode(chat_prefix_text(prompt),add_special_tokens=False)
    full_text=chat_prefix_text(prompt)+'\n'+reference
    full_ids=tokenizer.encode(full_text,add_special_tokens=False)
    start=0
    for a,b in zip(prefix_ids,full_ids):
        if a!=b: break
        start+=1
    if start<=0 or start>=len(full_ids): start=len(prefix_ids)
    if len(full_ids)<=start:
        full_ids=prefix_ids+tokenizer.encode('\n'+reference,add_special_tokens=False)
        start=len(prefix_ids)
    return full_ids,start

print('Loading model BF16...',flush=True); t0=time.time()
model,loading_info=AutoModelForCausalLM.from_pretrained(MODEL_PATH,token=HF_TOKEN,trust_remote_code=True,
    device_map={'':0},dtype=torch.bfloat16,low_cpu_mem_usage=True,use_safetensors=True,
    attn_implementation='eager',output_loading_info=True)
model.eval(); model.config.use_cache=False

def get_shards():
    for c in [Path(MODEL_PATH),Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2')]:
        if c.exists():
            s=sorted([p for p in c.glob('**/*.safetensors') if p.is_file() and p.stat().st_size>100*1024*1024])
            if s: return s
    try:
        from huggingface_hub import snapshot_download
        d=Path(snapshot_download(MODEL_ID,token=HF_TOKEN))
        return sorted([p for p in d.glob('*.safetensors') if p.stat().st_size>100*1024*1024])
    except: return []
shards=get_shards(); print(f'{len(shards)} shards',flush=True)
if shards:
    fused=0
    for sp in shards:
        try: sd=load_file(str(sp),device='cpu')
        except: continue
        with torch.no_grad():
            for li,layer in enumerate(model.model.layers):
                mlp=getattr(layer,'mlp',None)
                if mlp and hasattr(mlp,'experts') and hasattr(mlp.experts,'down_proj'):
                    for e in range(256):
                        dk=f'model.layers.{li}.mlp.experts.{e}.down_proj.weight'
                        gk=f'model.layers.{li}.mlp.experts.{e}.gate_proj.weight'
                        uk=f'model.layers.{li}.mlp.experts.{e}.up_proj.weight'
                        td=mlp.experts.down_proj
                        if dk in sd: mlp.experts.down_proj[e].copy_(sd[dk].to(device=td.device,dtype=td.dtype)); fused+=1
                        if gk in sd and uk in sd:
                            mlp.experts.gate_up_proj[e].copy_(torch.cat([sd[gk],sd[uk]],dim=0).to(device=td.device,dtype=td.dtype))
                bk=f'model.layers.{li}.mlp.experts.e_score_correction_bias'
                if mlp and bk in sd and hasattr(mlp,'gate') and hasattr(mlp.gate,'e_score_correction_bias') and mlp.gate.e_score_correction_bias is not None:
                    b=mlp.gate.e_score_correction_bias; b.copy_(sd[bk].to(device=b.device,dtype=b.dtype))
                if mlp and hasattr(mlp,'shared_experts'):
                    sh=mlp.shared_experts
                    for proj in ['down_proj','gate_proj','up_proj']:
                        sk=f'model.layers.{li}.mlp.shared_expert.{proj}.weight'
                        if sk in sd and hasattr(sh,proj):
                            w=getattr(sh,proj); ww=w.weight if hasattr(w,'weight') else w
                            ww.copy_(sd[sk].to(device=ww.device,dtype=ww.dtype))
        del sd; gc.collect()
    print(f'Fused {fused} expert weights',flush=True)
for p in model.parameters(): p.requires_grad_(False)
del loading_info; gc.collect(); torch.cuda.empty_cache()
enc=tokenizer(chat_prefix_text('What is 2+2?'),return_tensors='pt').to(DEV)
with torch.inference_mode():
    out=model.generate(**enc,max_new_tokens=32,do_sample=False)
    txt=tokenizer.decode(out[0,enc['input_ids'].shape[1]:],skip_special_tokens=True).strip()
print(f'Sanity: {txt[:80]}')
print(f'Loaded in {(time.time()-t0)/60:.1f}min | {sum(p.numel() for p in model.parameters()):,} params')


In [ ]:
# Cell 05 — Evaluators: GPQA generation + Code NLL

def extract_answer(text):
    if not text or not text.strip(): return ''
    s = text.strip()
    # 1. \boxed{X}
    idx = s.rfind(r'\boxed{')
    if idx != -1:
        content, depth = [], 0
        for c in s[idx+7:]:
            if c == '{': depth += 1; content.append(c)
            elif c == '}':
                if depth == 0: break
                depth -= 1; content.append(c)
            else: content.append(c)
        boxed = ''.join(content).strip().upper()
        bl = re.findall(r'\b([A-D])\b', boxed)
        if bl: return bl[-1]
    # 2. 'The answer is (X)'
    m = re.search(r'[Tt]he\s+answer\s+is\s*\(?([A-D])\)?', s)
    if m: return m.group(1).upper()
    # 3. Last (A)/(B)/(C)/(D)
    paren = re.findall(r'\(([A-D])\)', s)
    if paren: return paren[-1].upper()
    # 4. Last standalone letter
    tail = s[-80:] if len(s) > 80 else s
    ml = re.findall(r'(?<![a-zA-Z])([A-D])(?![a-zA-Z])', tail)
    if ml: return ml[-1].upper()
    return ''

@torch.inference_mode()
def evaluate_target(eval_model, df, tag='model', batch_size=EVAL_BATCH, max_tokens=EVAL_TOKENS):
    ev = df[(df['split']=='test')&(df['kind']=='target')].reset_index(drop=True)
    total = len(ev); correct = 0; results = []; t0 = time.time()
    old_ps = tokenizer.padding_side; tokenizer.padding_side = 'left'
    eval_model.eval()
    try:
        for si in range(0, total, batch_size):
            bdf = ev.iloc[si:si+batch_size]
            pfx = [chat_prefix_text(r.prompt) for r in bdf.itertuples(index=False)]
            enc = tokenizer(pfx, return_tensors='pt', padding=True, truncation=True, max_length=768).to(DEV)
            out = eval_model.generate(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                max_new_tokens=max_tokens, do_sample=False, use_cache=True,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            dec = tokenizer.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
            del out, enc
            for j, r in enumerate(bdf.itertuples(index=False)):
                ext = extract_answer(dec[j])
                target = str(r.target_answer).strip().upper()
                ic = 1.0 if ext == target else 0.0
                correct += int(ic)
                results.append({'method':tag,'example_id':r.example_id,'target':target,
                    'extracted':ext,'correct':ic,'output_len':len(dec[j]),'preview':dec[j][:150]})
            torch.cuda.empty_cache()
            done = min(si+batch_size, total)
            if done % (batch_size*3) == 0 or done == total:
                print(f'    [{done:03d}/{total}] {correct/done*100:4.1f}% | {time.time()-t0:.0f}s', flush=True)
    finally: tokenizer.padding_side = old_ps
    acc = correct / total
    print(f'  GPQA: {acc*100:.1f}% ({correct}/{total}) in {time.time()-t0:.0f}s', flush=True)
    return acc, pd.DataFrame(results)

# Code NLL control tasks
CTRL_TASKS = [
    ('Implement binary search.','def binary_search(arr,t):\n    lo,hi=0,len(arr)-1\n    while lo<=hi:\n        m=(lo+hi)//2\n        if arr[m]==t: return m\n        elif arr[m]<t: lo=m+1\n        else: hi=m-1\n    return -1'),
    ('Merge two sorted lists.','def merge(a,b):\n    r,i,j=[],0,0\n    while i<len(a) and j<len(b):\n        if a[i]<=b[j]: r.append(a[i]);i+=1\n        else: r.append(b[j]);j+=1\n    r.extend(a[i:]);r.extend(b[j:])\n    return r'),
    ('Implement a stack class.','class Stack:\n    def __init__(self): self._s=[]\n    def push(self,x): self._s.append(x)\n    def pop(self): return self._s.pop()\n    def peek(self): return self._s[-1]\n    def __len__(self): return len(self._s)'),
    ('Longest common subsequence.','def lcs(a,b):\n    m,n=len(a),len(b)\n    dp=[[0]*(n+1) for _ in range(m+1)]\n    for i in range(1,m+1):\n        for j in range(1,n+1):\n            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]+1\n            else: dp[i][j]=max(dp[i-1][j],dp[i][j-1])\n    return dp[m][n]'),
    ('Flatten nested list.','def flatten(lst):\n    r=[]\n    for x in lst:\n        if isinstance(x,list): r.extend(flatten(x))\n        else: r.append(x)\n    return r'),
    ('Fibonacci with memo.','def fib(n,m={}):\n    if n in m: return m[n]\n    if n<=1: return n\n    m[n]=fib(n-1,m)+fib(n-2,m)\n    return m[n]'),
    ('Quicksort.','def qsort(a):\n    if len(a)<=1: return a\n    p=a[len(a)//2]\n    return qsort([x for x in a if x<p])+[x for x in a if x==p]+qsort([x for x in a if x>p])'),
    ('Prime factors.','def factors(n):\n    f,d=[],2\n    while d*d<=n:\n        while n%d==0: f.append(d);n//=d\n        d+=1\n    if n>1: f.append(n)\n    return f'),
    ('BFS on graph.','from collections import deque\ndef bfs(g,s):\n    vis,q,r=set(),deque([s]),[]\n    vis.add(s)\n    while q:\n        n=q.popleft();r.append(n)\n        for nb in g.get(n,[]):\n            if nb not in vis: vis.add(nb);q.append(nb)\n    return r'),
    ('Roman to int.','def roman(s):\n    v={"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}\n    r=0\n    for i in range(len(s)):\n        if i+1<len(s) and v[s[i]]<v[s[i+1]]: r-=v[s[i]]\n        else: r+=v[s[i]]\n    return r'),
    ('Edit distance.','def edit(a,b):\n    m,n=len(a),len(b)\n    dp=[[0]*(n+1) for _ in range(m+1)]\n    for i in range(m+1): dp[i][0]=i\n    for j in range(n+1): dp[0][j]=j\n    for i in range(1,m+1):\n        for j in range(1,n+1):\n            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]\n            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])\n    return dp[m][n]'),
    ('Detect cycle in linked list.','def has_cycle(head):\n    s=f=head\n    while f and f.next:\n        s=s.next;f=f.next.next\n        if s is f: return True\n    return False'),
    ('Validate email.','import re\ndef valid_email(e):\n    return bool(re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$",e))'),
    ('Cache decorator.','from functools import wraps\ndef cache(f):\n    c={}\n    @wraps(f)\n    def w(*a):\n        if a not in c: c[a]=f(*a)\n        return c[a]\n    return w'),
    ('JSON weather response.','{\n  "city":"SF","temp":{"value":18.5,"unit":"C"},\n  "conditions":"cloudy","humidity":72\n}'),
    ('Context manager timer.','import time as _t\nfrom contextlib import contextmanager\n@contextmanager\ndef timer(l=""):\n    s=_t.perf_counter()\n    try: yield\n    finally: print(f"{l}:{_t.perf_counter()-s:.4f}s")'),
]
CTRL_CASES=[]
for p,r in CTRL_TASKS:
    ids,s=parse_case(p,r); CTRL_CASES.append({'ids':ids,'start':s})
print(f'{len(CTRL_CASES)} code control tasks')

@torch.inference_mode()
def evaluate_control(eval_model, df=None, tag='model', **kw):
    eval_model.eval(); t0=time.time()
    total_loss=0.0; total_tok=0
    for c in CTRL_CASES:
        ids=torch.tensor(c['ids'],dtype=torch.long,device=DEV).unsqueeze(0)
        s=c['start']
        with torch.autocast('cuda',dtype=torch.bfloat16):
            out=eval_model(input_ids=ids,use_cache=False)
        logits=out.logits[0,s-1:-1,:].float()
        targets=ids[0,s:]
        loss=F.cross_entropy(logits,targets,reduction='sum')
        total_loss+=float(loss.detach()); total_tok+=targets.shape[0]
        del out,ids
    nll=total_loss/max(1,total_tok)
    print(f'  Code NLL: {nll:.4f} ({total_tok} tok) in {time.time()-t0:.0f}s',flush=True)
    return nll, pd.DataFrame([{'method':tag,'nll':nll,'tokens':total_tok}])

print('Evaluators: GPQA gen (1024tok) + Code NLL (16 tasks)')


In [ ]:
# Cell 06 — Whitened Bases (norm-matched)
print('Auto-discovering attention modules...',flush=True)
_attn=set()
for name,mod in model.named_modules():
    if isinstance(mod,nn.Linear) and 'layers.' in name:
        suf=name.split('.')[-1]
        if 'attn' in name or 'self_attn' in name: _attn.add(suf)
        elif suf.endswith('_proj') and 'mlp' not in name and 'expert' not in name and 'gate' not in name: _attn.add(suf)
LORA_TARGET_MODULES=sorted(list(_attn)) if _attn else ['q_proj','k_proj','v_proj','o_proj']
print(f'LORA_TARGET_MODULES={LORA_TARGET_MODULES}',flush=True)

def harvest_cov(prompts,layers,max_samples=64):
    acts={l:{m:[] for m in LORA_TARGET_MODULES} for l in layers}; hooks=[]
    def mk(li,mn):
        def fn(mod,inp,out):
            if isinstance(inp,tuple) and len(inp)>0:
                x=inp[0].detach()
                if x.dim()==3: acts[li][mn].append(x[0,::4,:].float().cpu())
        return fn
    for name,mod in model.named_modules():
        for li in layers:
            if f'layers.{li}.' in name:
                for mn in LORA_TARGET_MODULES:
                    if mn in name and isinstance(mod,nn.Linear): hooks.append(mod.register_forward_hook(mk(li,mn)))
    with torch.no_grad():
        for p in prompts[:max_samples]:
            inp=tokenizer(chat_prefix_text(p),return_tensors='pt',truncation=True,max_length=256).to(DEV)
            model(**inp,use_cache=False); del inp
    for h in hooks: h.remove()
    cov={}
    for li in layers:
        for mn in LORA_TARGET_MODULES:
            vl=acts[li][mn]
            if vl:
                cat=torch.cat(vl,dim=0); cat=cat-cat.mean(0,keepdim=True)
                cov[(li,mn)]=(cat.T@cat)/max(1,cat.shape[0]-1)
    torch.cuda.empty_cache(); return cov

print('Harvesting control covariance (code)...',flush=True)
cov_ctrl=harvest_cov([p for p,_ in CTRL_TASKS],STRATIFIED_LAYERS,64)
print('Harvesting target covariance (STEM)...',flush=True)
tgt_prompts=BENCHMARK_DF[BENCHMARK_DF['kind']=='target']['prompt'].tolist()
cov_tgt=harvest_cov(tgt_prompts,STRATIFIED_LAYERS,128)

WHITENED_BASES={}
for key in cov_tgt:
    if key not in cov_ctrl: continue
    cc=cov_ctrl[key].to(DEV,dtype=torch.float32); ct=cov_tgt[key].to(DEV,dtype=torch.float32)
    ev_c,evec_c=torch.linalg.eigh(cc+0.05*torch.eye(cc.shape[0],device=DEV))
    inv_sqrt=1.0/torch.sqrt(torch.clamp_min(ev_c,0.0))
    Ginv=evec_c*inv_sqrt.unsqueeze(0)@evec_c.T
    St=Ginv@ct@Ginv; evt,evect=torch.linalg.eigh(St)
    Ur=evect[:,-LORA_RANK:]; A0=(Ur.T@Ginv).cpu().float()
    kn=math.sqrt(2.0/A0.shape[1])*math.sqrt(A0.shape[0]*A0.shape[1])
    A0=A0*(kn/A0.norm()); WHITENED_BASES[key]=A0
    del cc,ct,Ginv,St,evt,evect,Ur
torch.cuda.empty_cache()
norms=[float(v.norm()) for v in WHITENED_BASES.values()]
print(f'{len(WHITENED_BASES)} bases | norms: {min(norms):.2f}-{max(norms):.2f}')


In [ ]:
# Cell 07 — Pre-Tokenize Training Data
print('Pre-tokenizing...',flush=True); t0=time.time()
train_df=BENCHMARK_DF[BENCHMARK_DF['split']=='train'].reset_index(drop=True)
all_ids=[]; all_starts=[]
for r in train_df.itertuples(index=False):
    fids,s=parse_case(r.prompt,r.reference)
    if len(fids)>TRAIN_SEQ_LEN: fids=fids[:TRAIN_SEQ_LEN]; s=min(s,TRAIN_SEQ_LEN-1)
    all_ids.append(fids); all_starts.append(s)
mx=max(len(x) for x in all_ids); n=len(all_ids); pad=tokenizer.pad_token_id or 0
pids=torch.full((n,mx),pad,dtype=torch.long); pmask=torch.zeros((n,mx),dtype=torch.long)
plabels=torch.full((n,mx),-100,dtype=torch.long)
for i in range(n):
    L=len(all_ids[i]); pids[i,:L]=torch.tensor(all_ids[i]); pmask[i,:L]=1
    s=all_starts[i]; plabels[i,s:L]=torch.tensor(all_ids[i][s:])
TRAIN_IDS=pids.to(DEV); TRAIN_MASK=pmask.to(DEV); TRAIN_LABELS=plabels.to(DEV); N_TRAIN=n
print(f'{N_TRAIN} cases | max_len={mx} | {time.time()-t0:.1f}s')


In [ ]:
# Cell 08 — Base Model Evaluation
gc.collect(); torch.cuda.empty_cache()
print('='*60); print('BASE MODEL EVALUATION'); print('='*60)
BASE_TARGET_ACC, BASE_TARGET_DF = evaluate_target(model, BENCHMARK_DF, tag='base')
BASE_CTRL_NLL, BASE_CTRL_DF = evaluate_control(model, tag='base')
atomic_csv(BASE_TARGET_DF, RESULTS/'base_gpqa.csv')
atomic_csv(BASE_CTRL_DF, RESULTS/'base_nll.csv')
print(f'\nBASE: GPQA={BASE_TARGET_ACC*100:.1f}% | NLL={BASE_CTRL_NLL:.4f}')


In [ ]:
# Cell 09 — Three-Arm Experiment
from peft import LoraConfig, get_peft_model
for p in model.parameters(): p.requires_grad=False
peft_cfg=LoraConfig(r=LORA_RANK,lora_alpha=LORA_ALPHA,target_modules=LORA_TARGET_MODULES,
    layers_to_transform=STRATIFIED_LAYERS,bias='none',task_type='CAUSAL_LM')
peft_model=get_peft_model(model,peft_cfg)
peft_model.gradient_checkpointing_enable()
print(f'LoRA params: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,} | grad_ckpt=ON')

def reset_standard():
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
            sA.weight.requires_grad=True; sB.weight.requires_grad=True

def reset_geodesic():
    ap=0
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            mt=re.search(r'layers\.(\d+)\.',n); matched=False
            if mt:
                li=int(mt.group(1))
                for mn in LORA_TARGET_MODULES:
                    if mn in n and (li,mn) in WHITENED_BASES:
                        As=WHITENED_BASES[(li,mn)]
                        if sA.weight.shape==As.shape:
                            with torch.no_grad(): sA.weight.copy_(As.to(device=sA.weight.device,dtype=sA.weight.dtype)); sB.weight.zero_()
                            sA.weight.requires_grad=False; sB.weight.requires_grad=True; ap+=1; matched=True
            if not matched:
                with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
                sA.weight.requires_grad=True; sB.weight.requires_grad=True
    return ap

def reset_warm():
    ap=0
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            mt=re.search(r'layers\.(\d+)\.',n); matched=False
            if mt:
                li=int(mt.group(1))
                for mn in LORA_TARGET_MODULES:
                    if mn in n and (li,mn) in WHITENED_BASES:
                        As=WHITENED_BASES[(li,mn)]
                        if sA.weight.shape==As.shape:
                            with torch.no_grad(): sA.weight.copy_(As.to(device=sA.weight.device,dtype=sA.weight.dtype)); sB.weight.zero_()
                            sA.weight.requires_grad=True; sB.weight.requires_grad=True; ap+=1; matched=True
            if not matched:
                with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
                sA.weight.requires_grad=True; sB.weight.requires_grad=True
    return ap

def run_sft(seed):
    tp=[p for p in peft_model.parameters() if p.requires_grad]
    tc=sum(p.numel() for p in tp)
    opt=torch.optim.AdamW(tp,lr=TRAIN_LR,betas=(0.9,0.95),weight_decay=0.01)
    rng=np.random.default_rng(int(seed)); peft_model.train(); opt.zero_grad(set_to_none=True)
    t0=time.time(); step=0
    print(f'   SFT: {TRAIN_STEPS} steps, batch={TRAIN_BATCH}, {tc:,} params',flush=True)
    for ep in range(TRAIN_EPOCHS):
        order=rng.permutation(N_TRAIN)
        for bi in range(0,N_TRAIN,TRAIN_BATCH):
            idx=order[bi:bi+TRAIN_BATCH]
            if len(idx)<2: continue
            with torch.autocast('cuda',dtype=torch.bfloat16):
                out=peft_model(input_ids=TRAIN_IDS[idx],attention_mask=TRAIN_MASK[idx],use_cache=False)
                logits=out.logits.float()
                loss=F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),TRAIN_LABELS[idx][:,1:].reshape(-1),ignore_index=-100)
            if not torch.isfinite(loss): continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(tp,1.0)
            prog=step/max(1,TRAIN_STEPS-1)
            clr=LR_MIN+0.5*(TRAIN_LR-LR_MIN)*(1+math.cos(math.pi*prog))
            for pg in opt.param_groups: pg['lr']=clr
            opt.step(); opt.zero_grad(set_to_none=True); step+=1
            if step%32==0 or step==TRAIN_STEPS:
                print(f'      [{step:03d}/{TRAIN_STEPS}] Loss:{float(loss.detach()):.4f} LR:{clr:.2e} | {time.time()-t0:.0f}s',flush=True)
            if step>=TRAIN_STEPS: break
        if step>=TRAIN_STEPS: break
    del opt,tp; gc.collect(); torch.cuda.empty_cache()
    print(f'   Done in {time.time()-t0:.0f}s',flush=True)

arms=[('geodesic','Geodesic (A0 frozen)',reset_geodesic),
      ('warm','Warm LoRA (A0 trainable)',reset_warm),
      ('standard','Standard LoRA (random A)',reset_standard)]
all_results=[]; total_runs=len(arms)*len(SEEDS); run_idx=0
print(f'\n{"="*80}\nTHREE-ARM: {len(arms)} arms x {len(SEEDS)} seeds = {total_runs} runs\n{"="*80}')

for arm_name,arm_label,reset_fn in arms:
    for seed in SEEDS:
        run_idx+=1; tag=f'{arm_name}_s{seed}'
        print(f'\n{"="*80}\n[{run_idx}/{total_runs}] {arm_label} | Seed {seed}\n{"="*80}',flush=True)
        tr=time.time()
        n_reset=reset_fn()
        tp_now=sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
        print(f'   Reset: {n_reset if isinstance(n_reset,int) else "ok"} | {tp_now:,} trainable',flush=True)
        run_sft(seed); peft_model.eval()
        t_acc,t_df=evaluate_target(peft_model,BENCHMARK_DF,tag=tag)
        c_nll,c_df=evaluate_control(peft_model,tag=tag)
        atomic_csv(t_df,RESULTS/f'{tag}_gpqa.csv')
        t_gain=t_acc-BASE_TARGET_ACC; c_shift=abs(c_nll-BASE_CTRL_NLL); wall=time.time()-tr
        print(f'   RESULT: GPQA={t_acc*100:.1f}% (gain={t_gain:+.1%}) | NLL={c_nll:.4f} (shift={c_shift:.4f}) | {wall:.0f}s',flush=True)
        all_results.append({'arm':arm_name,'label':arm_label,'seed':seed,
            'gpqa_acc':t_acc,'gpqa_gain':t_gain,'code_nll':c_nll,'code_shift':c_shift,'wall_s':wall})

RESULTS_DF=pd.DataFrame(all_results)
atomic_csv(RESULTS_DF,RESULTS/'v23_all_results.csv')
print(f'\nALL {len(all_results)} RUNS COMPLETE!')


In [ ]:
# Cell 10 — Report
from IPython.display import display, Markdown
def bci(v,B=5000,s=2026):
    rng=np.random.default_rng(s)
    samps=[np.mean(rng.choice(v,len(v),replace=True)) for _ in range(B)]
    return {'mean':np.mean(v),'std':np.std(v),'lo':np.percentile(samps,2.5),'hi':np.percentile(samps,97.5)}
rows=[]
for arm,grp in RESULTS_DF.groupby('arm'):
    g=bci(grp['gpqa_gain'].values); c=bci(grp['code_shift'].values)
    rows.append({'Arm':grp['label'].iloc[0],'GPQA Gain':f"{g['mean']:+.1%}",
        'GPQA CI':f"[{g['lo']:+.1%},{g['hi']:+.1%}]",
        'NLL Shift':f"{c['mean']:.4f}",'NLL CI':f"[{c['lo']:.4f},{c['hi']:.4f}]"})
S=pd.DataFrame(rows); atomic_csv(S,RESULTS/'v23_summary.csv')
md=f'# v23 Results\n\nBase: GPQA={BASE_TARGET_ACC*100:.1f}% | NLL={BASE_CTRL_NLL:.4f}\n\n'
md+=S.to_markdown(index=False)+'\n\nLower NLL Shift = less forgetting. Higher GPQA Gain = better.\n'
with open(RESULTS/'v23_report.md','w') as f: f.write(md)
display(Markdown(md)); display(RESULTS_DF)


In [ ]:
# Cell 11 — Verify
for f in ['benchmark.csv','base_gpqa.csv','base_nll.csv','v23_all_results.csv']:
    assert (RESULTS/f).exists(), f'Missing: {f}'
print('v23 COMPLETE. All files verified.')
